### Load Dependencies and Data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import sys
import yaml
import numpy as np

Add src folder to sys.path so Python can find your package

In [ ]:
ROOT = Path.cwd().parent  # project root directory when working interactively
sys.path.append(str(ROOT / "src"))

Import the root folder of the project : house_prices_tensorflow

In [ ]:
from pkg_house_prices.utils.project_root import PROJECT_ROOT

Import the data

In [ ]:
from pkg_house_prices.data.data_loader import train, test

In [2]:
from pkg_house_prices.utils.config import CONFIG  # your YAML loader
from pkg_house_prices.utils.helpers import read_config
import pandas as pd

# Logging paths
train_path = read_config("data", "train")
test_path = read_config("data", "test")
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

2026-01-25 18:46:34,401 - house_prices - INFO - read_config() - Reading config keys: ('data', 'train')
2026-01-25 18:46:34,401 - house_prices - INFO - read_config() - Project Root is D:\Joseph\Knowledge\Kaggle\house_prices_tensorflow
2026-01-25 18:46:34,401 - house_prices - INFO - read_config() - Reading config keys: ('data', 'test')
2026-01-25 18:46:34,401 - house_prices - INFO - read_config() - Project Root is D:\Joseph\Knowledge\Kaggle\house_prices_tensorflow


In [6]:
print(test.columns)
print(train.columns)

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

### Data Exploration - Data types and missing values

In [ ]:
# Determine the data types of each column in the training data
print(train.info())

In [ ]:
# Return columns with object data type
object_cols = train.select_dtypes(include=["object"]).columns
print("\nColumns with object data type:")
print(object_cols)
# Display first few entries of these object columns
print("\nFirst few entries of object columns:")
print(train[object_cols].head())

In [ ]:
# Convert object data types into string data types for easier handling
for col in object_cols:
    train[col] = train[col].astype("string")
    test[col] = test[col].astype("string")

In [ ]:
# Count the number of missing values in each column and print the result
missing_counts = train.isnull().sum()
print("\nMissing values in each column of the training data:")
print(missing_counts[missing_counts > 0])
print(
    "\nTotal number of columns with missing values is",
    len(missing_counts[missing_counts > 0]),
    " out of ",
    len(train.columns),
    ".",
)

##### Drop Missing Values and Save New Data

In [ ]:
# Drop the columns with missing values for now
train = train.drop(columns=missing_counts[missing_counts > 0].index)
test = test.drop(columns=missing_counts[missing_counts > 0].index)
print("\nAfter dropping, the training data now has columns:")
print(train.info())

# save cleaned datasets
train.to_csv(PROJECT_ROOT / "data/processed/train_cleaned.csv", index=False)
test.to_csv(PROJECT_ROOT / "data/processed/test_cleaned.csv", index=False)

Summary: 
1. Columns with object datatypes were transformed to string. 
2. All columns with missing values are dropped since they only cover less than 25% of the actual number of variables. It is assumed that the remaining variables are sufficent to predict the house prices. 

### Plot the Data

In [ ]:
# Plot the distribution of the target variable 'SalePrice'
plt.hist(train["SalePrice"], bins=30, edgecolor="black")

In [ ]:
# From the plot above, the house price data appears to be right-skewed. A log transformation can help normalize the distribution.
plt.hist(np.log1p(train["SalePrice"]), bins=30, edgecolor="black")

In [ ]:
# Plot the histogram of explanatory variables in n by 4 grid of subplots
num_cols = len(train.columns) - 1  # Exclude target variable
n_rows = (num_cols + 3) // 4  # Calculate number of rows
fig, axes = plt.subplots(n_rows, 4, figsize=(20, 5 * n_rows))

for i, col in enumerate(train.columns):
    if col == "SalePrice":
        continue  # Skip the target variable
    ax = axes[i // 4, i % 4]
    ax.hist(train[col].dropna(), bins=30, edgecolor="black")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Identify variables with element having count more than 1400 (95.9%)
for col in train.columns:
    value_counts = train[col].value_counts()
    high_count_values = value_counts[value_counts > 1400]
    if not high_count_values.empty:
        print(f"\nColumn '{col}' has the following values with counts greater than 1400:")
        print(high_count_values)

In [ ]:
# return the variables with discrete numerical values that may be categorical
discrete_numerical_cols = [
    col for col in train.select_dtypes(include=["int64", "float64"]).columns if train[col].nunique() < 15
]
print("\nDiscrete numerical columns that may be categorical:")
# print in column format
for x in discrete_numerical_cols:
    print(x)

In [ ]:
# Print the values of LowQualFinSF to see why it has so many zeros
print("\nValues of LowQualFinSF:")
print(train["LowQualFinSF"].value_counts())
# It appears that most houses have no low quality finished square feet, leading to many zeros in this column.

#### SCATTERPLOTS - NUMERICAL VARIABLES

In [ ]:
# Create scatter plots in n by 4 grid with regression lines and correlation for numerical variables against SalePrice
numerical_cols = train.select_dtypes(include=["int64", "float64"]).columns.drop("SalePrice")
n_rows = (len(numerical_cols) + 3) // 4  # Calculate number of rows
fig, axes = plt.subplots(n_rows, 4, figsize=(20, 5 * n_rows))
for i, col in enumerate(numerical_cols):
    ax = axes[i // 4, i % 4]
    sns.regplot(x=train[col], y=train["SalePrice"], ax=ax, scatter_kws={"alpha": 0.5})
    corr = train[col].corr(train["SalePrice"])
    ax.set_title(f"{col} vs SalePrice (corr={corr:.2f})")
plt.tight_layout()
plt.show()

#### BOXPLOTS - CATEGORICAL VARIABLES

In [ ]:
# Create box plots for categorical variables against SalePrice, order the plots according to increasing  median
categorical_cols = train.select_dtypes(include=["string"]).columns
n_rows = (len(categorical_cols) + 3) // 4  # Calculate number of rows
fig, axes = plt.subplots(n_rows, 4, figsize=(20, 5 * n_rows))

for i, col in enumerate(categorical_cols):
    ax = axes[i // 4, i % 4]
    medians = train.groupby(col)["SalePrice"].median().sort_values()
    sns.boxplot(x=train[col], y=train["SalePrice"], ax=ax, order=medians.index)
    ax.set_title(f"{col} vs SalePrice")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Finalize the cleaned datasets by translating objects and strings to numeirical values
# For categorical variables, use one-hot encoding
train = pd.get_dummies(train, drop_first=True)
test = pd.get_dummies(test, drop_first=True)